# ⚡ パフォーマンスメトリクス分析

レイテンシ・スループット・エラーパターンを分析します。

**データソース:** `messages` / `routing_decisions` / `threads` テーブル

In [ ]:
import utils
from utils import qdf, scalar, ROLE_JA, PALETTE
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

## 1. 全体レスポンスタイム分布

In [ ]:
rt = qdf("""
    SELECT execution_time AS ms, agent_role
    FROM messages
    WHERE role='assistant' AND execution_time > 0
      AND execution_time < 300000  -- 5分超は外れ値除外
""")

if utils.check_data(rt, 'レスポンスタイムデータ'):
    ms = rt['ms'].dropna()
    p50  = np.percentile(ms, 50)
    p95  = np.percentile(ms, 95)
    p99  = np.percentile(ms, 99)

    print('=== レスポンスタイム統計 ===')
    print(f'  サンプル数 : {len(ms):>6}')
    print(f'  最小       : {ms.min():>8.0f} ms')
    print(f'  平均       : {ms.mean():>8.0f} ms')
    print(f'  中央値(P50): {p50:>8.0f} ms')
    print(f'  P95        : {p95:>8.0f} ms')
    print(f'  P99        : {p99:>8.0f} ms')
    print(f'  最大       : {ms.max():>8.0f} ms')

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].hist(ms, bins=30, color=PALETTE[0], edgecolor='white', alpha=0.85)
    axes[0].axvline(p50, color='royalblue', linestyle='--', lw=1.5, label=f'P50 {p50:.0f}ms')
    axes[0].axvline(p95, color='darkorange', linestyle='--', lw=1.5, label=f'P95 {p95:.0f}ms')
    axes[0].axvline(p99, color='crimson',    linestyle='--', lw=1.5, label=f'P99 {p99:.0f}ms')
    axes[0].set_title('レスポンスタイム分布')
    axes[0].set_xlabel('ミリ秒 (ms)')
    axes[0].set_ylabel('件数')
    axes[0].legend()

    # 対数スケール版
    axes[1].hist(np.log10(ms.clip(lower=1)), bins=30, color=PALETTE[1], edgecolor='white', alpha=0.85)
    axes[1].set_title('レスポンスタイム分布（対数スケール）')
    axes[1].set_xlabel('log10(ms)')
    axes[1].set_ylabel('件数')
    ticks = [1, 10, 100, 1000, 10000, 100000]
    axes[1].set_xticks([np.log10(t) for t in ticks])
    axes[1].set_xticklabels([f'{t:,}ms' for t in ticks], fontsize=8)

    plt.tight_layout()
    plt.show()

## 2. ロール別 レスポンスタイム箱ひげ図

In [ ]:
if utils.check_data(rt, 'ロール別レスポンスタイム') and 'agent_role' in rt.columns:
    box_data = [
        rt[rt['agent_role'] == r]['ms'].dropna().values
        for r in rt['agent_role'].unique()
        if len(rt[rt['agent_role'] == r]) >= 2
    ]
    box_labels = [
        ROLE_JA.get(r, r)
        for r in rt['agent_role'].unique()
        if len(rt[rt['agent_role'] == r]) >= 2
    ]

    if box_data:
        fig, ax = plt.subplots(figsize=(12, 5))
        bp = ax.boxplot(box_data, labels=box_labels, patch_artist=True, vert=True,
                        flierprops=dict(marker='o', markersize=3, alpha=0.5))
        colors = [PALETTE[i % len(PALETTE)] for i in range(len(box_data))]
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        ax.set_title('ロール別 レスポンスタイム分布')
        ax.set_ylabel('ミリ秒 (ms)')
        ax.tick_params(axis='x', rotation=20)
        ax.set_yscale('log')
        ax.set_ylabel('ミリ秒 (ms, 対数スケール)')
        plt.tight_layout()
        plt.show()
    else:
        print('ℹ️  2件以上のデータがあるロールが揃うと箱ひげ図が表示されます')

## 3. 時間帯別スループット

In [ ]:
hourly = qdf("""
    SELECT EXTRACT(HOUR FROM created_at)::int AS hour,
           COUNT(*) AS cnt
    FROM messages
    WHERE role='user'
    GROUP BY hour
    ORDER BY hour
""")

if utils.check_data(hourly, '時間帯別データ'):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(hourly['hour'], hourly['cnt'], color=PALETTE[0], width=0.8, edgecolor='white')
    ax.set_title('時間帯別 ユーザーメッセージ数')
    ax.set_xlabel('時間 (時)')
    ax.set_ylabel('件数')
    ax.set_xticks(range(0, 24))
    plt.tight_layout()
    plt.show()

## 4. スレッド長の分布

In [ ]:
thread_len = qdf("""
    SELECT t.id, t.title,
           COUNT(m.id) AS msg_count,
           COUNT(m.id) FILTER (WHERE m.role='user')      AS user_msgs,
           COUNT(m.id) FILTER (WHERE m.role='assistant') AS bot_msgs,
           EXTRACT(EPOCH FROM (MAX(m.created_at) - MIN(m.created_at)))/60 AS duration_min
    FROM threads t
    JOIN messages m ON m.thread_id = t.id
    WHERE NOT t.is_archived
    GROUP BY t.id, t.title
    HAVING COUNT(m.id) > 0
    ORDER BY msg_count DESC
""")

if utils.check_data(thread_len, 'スレッド長データ'):
    print(f'=== スレッド統計 ===')
    print(f'  スレッド数   : {len(thread_len)}')
    print(f'  平均会話数   : {thread_len["msg_count"].mean():.1f} 件')
    print(f'  最長スレッド : {thread_len["msg_count"].max()} 件  ─  {thread_len.iloc[0]["title"][:40]}')

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # スレッド長ヒストグラム
    axes[0].hist(thread_len['msg_count'], bins=min(15, len(thread_len)),
                 color=PALETTE[2], edgecolor='white', alpha=0.85)
    axes[0].set_title('スレッド長分布')
    axes[0].set_xlabel('メッセージ数')
    axes[0].set_ylabel('スレッド数')

    # 長いスレッド TOP10
    top10 = thread_len.head(10).copy()
    top10['label'] = top10['title'].str[:30]
    axes[1].barh(top10['label'], top10['msg_count'], color=PALETTE[1], edgecolor='white')
    axes[1].set_title('会話数 TOP10 スレッド')
    axes[1].set_xlabel('メッセージ数')
    axes[1].invert_yaxis()

    plt.tight_layout()
    plt.show()

## 5. ルーティング決定の分析（`routing_decisions` が蓄積後に有効）

In [ ]:
rd = qdf("""
    SELECT selected_role, intent_type, complexity,
           ROUND(confidence::numeric, 3) AS confidence,
           ROUND(total_latency_ms::numeric, 0) AS latency_ms,
           created_at
    FROM routing_decisions
    ORDER BY created_at DESC
    LIMIT 200
""")

if utils.check_data(rd, 'ルーティング決定データ'):
    print(f'ルーティング決定数: {len(rd)}')

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))

    # ロール別選択回数
    rd['role_ja'] = rd['selected_role'].map(lambda r: ROLE_JA.get(r, r))
    rd['role_ja'].value_counts().plot(kind='barh', ax=axes[0,0], color=PALETTE[0])
    axes[0,0].set_title('選択ロール分布')

    # Intent型分布
    rd['intent_type'].value_counts().plot(kind='pie', ax=axes[0,1],
        autopct='%1.1f%%', colors=PALETTE)
    axes[0,1].set_title('Intent 型分布')
    axes[0,1].set_ylabel('')

    # Complexity 分布
    rd['complexity'].value_counts().plot(kind='bar', ax=axes[1,0], color=PALETTE[2])
    axes[1,0].set_title('Complexity 分布')
    axes[1,0].tick_params(axis='x', rotation=20)

    # 信頼度分布
    rd['confidence'].hist(bins=20, ax=axes[1,1], color=PALETTE[3], edgecolor='white')
    axes[1,1].set_title('ルーティング信頼度分布')
    axes[1,1].set_xlabel('confidence')

    plt.tight_layout()
    plt.show()
else:
    print('ℹ️  routing_decisions はチャット利用時に自動記録されます')
    print('    現在はまだデータがありません')